Multimodel Architecture - Routing Workflow 

In [ ]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
openai_client = OpenAI(api_key=openai_api_key)
deepseek_client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
groq_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

In [ ]:
MODEL_REGISTRY = {
    "gpt-5-nano": {
        "provider": "openai",
        "strength": "general",
        "cost": "low"
    },
    "gpt-5-mini": {
        "provider": "openai",
        "strength": "reasoning",
        "cost": "medium"
    },
    
    "deepseek-chat": {
        "provider": "deepseek",
        "strength": "coding",
        "cost": "low"
    },
    "gemini-2.5-flash": {
        "provider": "google",
        "strength": "general",
        "cost": "low"
    }
}


ROUTER AGENT

In [ ]:
def classify_task(user_input):
    router_prompt = f"""
    Classify the task into ONE of these categories:
    - coding
    - creative_writing
    - quantitative_reasoning
    - strategic_analysis
    - simple_general

    Respond ONLY with the category name.

    User request:
    {user_input}
    """

    response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": router_prompt}],
    )

    return response.choices[0].message.content.strip()


ROUTING LOGIC

In [ ]:
def select_model(task_type):
    if task_type == "coding":
        return "deepseek-chat"
    elif task_type == "creative_writing":
        return "gemini-2.5-flash"
    elif task_type == "quantitative_reasoning":
        return "gpt-5-mini"
    elif task_type == "strategic_analysis":
        return "gpt-5-mini"
    else:
        return "gpt-5-nano"


EXECUTION LAYER

In [ ]:
def call_model(model_name, user_input):

    if model_name in ["gpt-5-nano", "gpt-5-mini"]:
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": user_input}],
        )
        return response.choices[0].message.content

    elif model_name == "deepseek-chat":
        response = deepseek_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": user_input}],
        )
        return response.choices[0].message.content

    elif model_name == "gemini-2.5-flash":
        response = gemini_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": user_input}],
        )
        return response.choices[0].message.content

    else:
        raise ValueError("Unknown model")


In [ ]:
def route_and_execute(user_input):

    print("Classifying task...")
    task_type = classify_task(user_input)
    print("Task type:", task_type)

    model_name = select_model(task_type)
    print("Selected model:", model_name)

    print("Executing")
    answer = call_model(model_name, user_input)

    return answer


TESTING

In [ ]:
user_question = "Write a Python function to calculate factorial recursively."

result = route_and_execute(user_question)

print("\nFinal Answer:\n")
print(result)


In [ ]:
user_question = "If a company reduces prices by 12% and sales volume increases by 18%, under what conditions does total revenue increase?"

result = route_and_execute(user_question)

print("\nFinal Answer:\n")
print(result)

In [ ]:
user_question = "Write a 150 word essay on AI impacting creative industry"

result = route_and_execute(user_question)

print("\nFinal Answer:\n")
print(result)

In [ ]:
user_question = "A mid-sized consulting firm is losing clients due to AI automation. Propose a structured turnaround strategy."

result = route_and_execute(user_question)

print("\nFinal Answer:\n")
print(result)

In [ ]:
user_question = "A mid-sized consulting firm is losing clients due to AI automation. Propose a structured turnaround strategy."

result = route_and_execute(user_question)

print("\nFinal Answer:\n")
print(result)